In [1]:
import json
import math
from pathlib import Path

import numpy as np
import yaml
from sympy.physics.quantum.cg import CG

EPS = 1e-8

# Functions

In [2]:
# -----------------------------------------------------------------------------
# Minimal, self-contained copies/adaptations of the TF-PWA math used on the
# Psi(4040) chain. These functions intentionally do not import tf_pwa.
# -----------------------------------------------------------------------------


def dot3(a, b):
    return np.sum(a * b, axis=-1)


def norm3(a):
    return np.linalg.norm(a, axis=-1)


def unit(v):
    n = np.linalg.norm(v, axis=-1, keepdims=True)
    return v / n


def cross_unit(a, b):
    # Copied from TF-PWA Vector3.cross_unit in spirit: if the cross product is
    # degenerate, bias the second vector before normalizing.
    cro = np.cross(a, b)
    norm = np.linalg.norm(cro, axis=-1, keepdims=True)
    mask = norm < EPS
    bias_other = np.ones_like(norm) + b
    cro = np.where(mask, np.cross(a, bias_other), cro)
    return unit(cro)


def angle_from(v, x_axis, y_axis):
    # Angle of vector v measured in the coordinate basis (x_axis, y_axis).
    return np.arctan2(dot3(v, y_axis), dot3(v, x_axis))


def angle_zx_z_getx(z1, x1, z2):
    # Adapted from TF-PWA EulerAngle.angle_zx_z_getx.
    # Physical meaning: construct the Euler rotation from an old frame (z1,x1)
    # to a new helicity frame whose z-axis follows a daughter momentum z2.
    u_z1 = unit(z1)
    u_z2 = unit(z2)
    u_y1 = cross_unit(z1, x1)
    u_x1 = cross_unit(u_y1, z1)
    u_yr = cross_unit(z1, z2)
    u_xr = cross_unit(u_yr, z1)
    alpha = angle_from(u_xr, u_x1, u_y1)
    beta = angle_from(u_z2, u_z1, u_xr)
    gamma = np.zeros_like(beta)
    u_x2 = cross_unit(u_yr, u_z2)
    return {"alpha": alpha, "beta": beta, "gamma": gamma}, u_x2


def invariant_mass(p4):
    # TF-PWA LorentzVector.M with metric (+,-,-,-).
    m2 = p4[..., 0] ** 2 - np.sum(p4[..., 1:] ** 2, axis=-1)
    return np.sqrt(np.abs(m2))


def boost_vector(p4):
    return p4[..., 1:] / p4[..., 0:1]


def boost(p4, beta):
    # Adapted from TF-PWA LorentzVector.boost.
    beta2 = np.sum(beta * beta, axis=-1)
    gamma = 1.0 / np.sqrt(1.0 - beta2)
    bp = np.sum(beta * p4[..., 1:], axis=-1)
    gamma2 = np.where(beta2 > EPS, (gamma - 1.0) / beta2, 0.0)
    spatial = p4[..., 1:]
    spatial = spatial + gamma2[..., None] * bp[..., None] * beta
    spatial = spatial + gamma[..., None] * p4[..., 0:1] * beta
    energy = (gamma * (p4[..., 0] + bp))[..., None]
    return np.concatenate([energy, spatial], axis=-1)


def rest_vector(core_p4, other_p4):
    # Boost other_p4 into the rest frame of core_p4.
    return boost(other_p4, -boost_vector(core_p4))


def get_relative_p2(m0, m1, m2):
    # Two-body breakup momentum squared, copied from TF-PWA formula.get_relative_p2.
    return ((m0 * m0 - (m1 + m2) ** 2) * (m0 * m0 - (m1 - m2) ** 2)) / (4 * m0 * m0)


def bprime_polynomial(l, z):
    # Blatt-Weisskopf polynomials used by TF-PWA up to the orders needed here.
    coeff = {
        0: [1.0],
        1: [1.0, 1.0],
        2: [1.0, 3.0, 9.0],
        3: [1.0, 6.0, 45.0, 225.0],
    }
    return np.polyval(coeff[int(l)], z)


def bprime_q2(l, q2, q02, d=3.0):
    # Adapted from TF-PWA breit_wigner.Bprime_q2.
    z0 = q02 * d**2
    z = q2 * d**2
    ratio = bprime_polynomial(l, z0) / bprime_polynomial(l, z)
    return np.sqrt(np.where(ratio > 0, ratio, 1.0))


def barrier_factor2(l, mass, q2, q02, d=3.0, barrier_factor_norm=True):
    # Adapted from HelicityDecay.get_barrier_factor2 for this chain.
    # Physical meaning: centrifugal-barrier factor q^L B'_L(q,q0,d), normalized by q0^L.
    tmp = q2 ** (l / 2) * bprime_q2(l, q2, q02, d)
    if barrier_factor_norm:
        tmp = tmp / np.abs(q02) ** (l / 2)
    return tmp.reshape(-1, 1)


def gamma_running(m, gamma0, q, q0, l, m0, d=3.0):
    # Adapted from TF-PWA breit_wigner.Gamma.
    qq0 = np.where(q0 > 1e-15, (q / q0) ** (2 * l + 1), 1.0)
    mm0 = m0 / m
    bp = (np.sqrt(bprime_polynomial(l, (q0 * d) ** 2)) / np.sqrt(bprime_polynomial(l, (q * d) ** 2))) ** 2
    return gamma0 * qq0 * mm0 * bp


def bwr(m, m0, gamma0, q, q0, l, d=3.0):
    # Adapted from TF-PWA breit_wigner.BWR.
    # Physical meaning: relativistic Breit-Wigner propagator with running width.
    gamma_m = gamma_running(m, gamma0, q, q0, l, m0, d)
    x = m0 * m0 - m * m
    y = m0 * gamma_m
    denom = x * x + y * y
    return x / denom + 1j * y / denom


def small_d_weight(j2):
    # Copied/adapted from TF-PWA dfun.small_d_weight. j2 means 2*j.
    ret = np.zeros((j2 + 1, j2 + 1, j2 + 1))

    def half_factorial(x):
        return math.factorial(x >> 1)

    for m in range(-j2, j2 + 1, 2):
        for n in range(-j2, j2 + 1, 2):
            for k in range(max(0, n - m), min(j2 - m, j2 + n) + 1, 2):
                ell = (2 * k + (m - n)) // 2
                sign = (-1) ** ((k + m - n) // 2)
                val = sign * math.sqrt(
                    half_factorial(j2 + m)
                    * half_factorial(j2 - m)
                    * half_factorial(j2 + n)
                    * half_factorial(j2 - n)
                )
                val /= (
                    half_factorial(j2 - m - k)
                    * half_factorial(j2 + n - k)
                    * half_factorial(k + m - n)
                    * half_factorial(k)
                )
                ret[ell][(m + j2) // 2][(n + j2) // 2] = val
    return ret


def small_d_matrix(theta, j2):
    theta = np.asarray(theta)
    powers = np.arange(0, j2 + 1).reshape(1, -1)
    half_theta = 0.5 * theta.reshape(-1, 1)
    sc = (np.sin(half_theta) ** powers) * (np.cos(half_theta) ** (j2 - powers))
    weights = small_d_weight(j2).reshape(j2 + 1, (j2 + 1) * (j2 + 1))
    return (sc @ weights).reshape(-1, j2 + 1, j2 + 1)


def d_matrix_conj(alpha, beta, gamma, j2):
    # Adapted from TF-PWA dfun.D_matrix_conj.
    # Physical meaning: conjugated Wigner D matrix D^J*(alpha,beta,gamma).
    m = np.arange(-j2 / 2, j2 / 2 + 1, 1).reshape(1, -1)
    d_small = small_d_matrix(beta, j2)
    exp_alpha = np.exp(1j * alpha.reshape(-1, 1) * m).reshape(-1, j2 + 1, 1)
    exp_gamma = np.exp(1j * gamma.reshape(-1, 1) * m).reshape(-1, 1, j2 + 1)
    return exp_alpha * exp_gamma * d_small.astype(complex)


def dfun_delta_v2(d, ja, la, lb, lc=(0,)):
    # Adapted from TF-PWA dfun.Dfun_delta_v2.
    ln = int(2 * ja + 1 + 0.1)
    idx = []
    max_idx = ln * ln
    for la_i in la:
        for lb_i in lb:
            for lc_i in lc:
                delta = lb_i - lc_i
                if abs(delta) <= ja:
                    idx.append(int((la_i + ja) * ln + delta + ja + 0.1))
                else:
                    idx.append(max_idx)
    flat = d.reshape(-1, ln * ln)
    padded = np.pad(flat, ((0, 0), (0, 1)))
    return padded[:, idx].reshape(-1, len(la), len(lb), len(lc))


def get_d_matrix_lambda(angle, ja, la, lb, lc=None):
    # Adapted from TF-PWA dfun.get_D_matrix_lambda.
    d = d_matrix_conj(angle["alpha"], angle["beta"], angle.get("gamma", np.zeros_like(angle["beta"])), int(2 * ja + 0.1))
    if lc is None:
        return dfun_delta_v2(d, ja, la, lb, (0,)).reshape(-1, len(la), len(lb))
    return dfun_delta_v2(d, ja, la, lb, lc)


def cg_coef(j1, j2, m1, m2, j, m):
    # Same convention as TF-PWA cg.cg_coef when SymPy is available.
    return float(CG(j1, m1, j2, m2, j, m).doit().evalf())


def cg_matrix(ja, jb, jc, ls_list, out_spins):
    # Adapted from HelicityDecay._get_cg_matrix.
    # Physical meaning: LS-to-helicity transformation matrix.
    ret = np.zeros((len(ls_list), len(out_spins[0]), len(out_spins[1])))
    for i, (ell, spin) in enumerate(ls_list):
        for ib, lambda_b in enumerate(out_spins[0]):
            for ic, lambda_c in enumerate(out_spins[1]):
                ret[i, ib, ic] = (
                    math.sqrt(2 * ell + 1)
                    / math.sqrt(2 * ja + 1)
                    * cg_coef(jb, jc, lambda_b, -lambda_c, spin, lambda_b - lambda_c)
                    * cg_coef(ell, spin, 0, lambda_b - lambda_c, ja, lambda_b - lambda_c)
                )
    return ret


def helicity_decay_amp(core_j, out_js, core_spins, out_spins, ls_list, g_ls, angle, mass_core, q2, q02, has_barrier=True, barrier_norm=True):
    # Adapted from HelicityDecay.get_amp -> get_helicity_amp -> get_ls_amp -> get_D_matrix_term.
    # Physical meaning: tensor amplitude for one two-body decay vertex.
    if has_barrier:
        ell = ls_list[0][0]
        bf = barrier_factor2(ell, mass_core, q2, q02, d=3.0, barrier_factor_norm=barrier_norm)
        m_dep = np.array([[g_ls]], dtype=complex) * bf.astype(complex)
    else:
        m_dep = np.array([[g_ls]], dtype=complex)

    cg = cg_matrix(core_j, out_js[0], out_js[1], ls_list, out_spins).astype(complex)
    h = np.sum(m_dep.reshape(-1, len(ls_list), 1, 1) * cg.reshape(len(ls_list), len(out_spins[0]), len(out_spins[1])), axis=1)
    h = h.reshape(-1, 1, len(out_spins[0]), len(out_spins[1]))
    d_conj = get_d_matrix_lambda(angle, core_j, core_spins, out_spins[0], out_spins[1])
    return h * d_conj.reshape(-1, len(core_spins), len(out_spins[0]), len(out_spins[1]))


def compute_chain_boosts(particle_p4, chain):
    # Adapted from TF-PWA cal_chain_boost.
    # Physical meaning: boost daughters step-by-step into each mother rest frame.
    particle_set = {name for _, outs in chain for name in outs}
    core_decay_map = {}
    part_data = {}
    pending = list(chain)
    while pending:
        extra = []
        for core, outs in pending:
            if core == "Bp":
                p_rest = particle_p4[core]
                part_data[core] = {"rest_p": {}}
                for out in outs:
                    core_decay_map[out] = core
                    part_data[core]["rest_p"][out] = rest_vector(p_rest, particle_p4[out])
                    particle_set.discard(out)
                for other in list(particle_set):
                    part_data[core]["rest_p"][other] = rest_vector(p_rest, particle_p4[other])
            elif core in core_decay_map:
                parent = core_decay_map[core]
                p_rest = part_data[parent]["rest_p"][core]
                part_data[core] = {"rest_p": {}}
                for out in outs:
                    core_decay_map[out] = core
                    part_data[core]["rest_p"][out] = rest_vector(p_rest, part_data[parent]["rest_p"][out])
                    particle_set.discard(out)
                for other in list(particle_set):
                    part_data[core]["rest_p"][other] = rest_vector(p_rest, part_data[parent]["rest_p"][other])
            else:
                extra.append((core, outs))
        pending = extra
    return part_data


def calculate_helicity_angles(particle_p4, chain):
    # Adapted from TF-PWA cal_helicity_angle for this chain.
    # Physical meaning: construct the helicity angles used in Wigner-D functions.
    part_data = compute_chain_boosts(particle_p4, chain)
    set_x = {"Bp": np.array([[1.0, 0.0, 0.0]])}
    set_z = {"Bp": np.array([[0.0, 0.0, 1.0]])}
    angles = {}
    for core, outs in chain:
        angles[core] = {}
        bias = -np.pi
        for out in outs:
            z2 = part_data[core]["rest_p"][out][..., 1:]
            ang, x_axis = angle_zx_z_getx(set_z[core], set_x[core], z2)
            set_x[out] = x_axis
            set_z[out] = z2
            ang["alpha"] = (ang["alpha"] - bias) % (2 * np.pi) + bias
            bias -= np.pi
            angles[core][out] = ang
    return angles


def format_complex(z):
    z = np.asarray(z).reshape(-1)[0]
    sign = "+" if z.imag >= 0 else "-"
    return f"{z.real:.16g} {sign} {abs(z.imag):.16g}j"

# Load Input

In [3]:
# -----------------------------------------------------------------------------
# Leaf inputs: files plus the same hardcoded four-vectors as tf_pwa_analysis_Gemini.py
# -----------------------------------------------------------------------------

analysis_dir = Path.cwd()
print(f"Notebook working directory: {analysis_dir}")

with open(analysis_dir / "config_a.yml", "r", encoding="utf-8") as f:
    config_yml = yaml.safe_load(f)

with open(analysis_dir / "Resonances.yml", "r", encoding="utf-8") as f:
    resonances_yml = yaml.safe_load(f)

with open(analysis_dir / "final_params_full.json", "r", encoding="utf-8") as f:
    params = json.load(f)["value"]

print("Loaded config_a.yml, Resonances.yml, and final_params_full.json")

# Hardcoded event four-vectors from tf_pwa_analysis_Gemini.py, order (E, px, py, pz).
p4 = {
    "D": np.array([[2.0452, -0.1467, 0.2235, -0.7847]], dtype=float),
    "D0": np.array([[2.2606, 0.2284, -0.3689, 1.2019]], dtype=float),
    "K": np.array([[0.7718, -0.0873, 0.1803, -0.5584]], dtype=float),
    "pi": np.array([[0.2017, 0.0056, -0.0349, 0.1413]], dtype=float),
}

# Extra event-level selector used by the local C(...) particle wrappers.
c_selector = np.array([-1.0])

# Nominal masses from the YAML model plus Psi(4040) mass/width from the JSON parameters.
nominal_mass = {
    "Bp": float(config_yml["particle"]["$top"]["Bp"]["mass"]),
    "D": float(config_yml["particle"]["$finals"]["D"]["mass"]),
    "K": float(config_yml["particle"]["$finals"]["K"]["mass"]),
    "D0": float(config_yml["particle"]["$finals"]["D0"]["mass"]),
    "pi": float(config_yml["particle"]["$finals"]["pi"]["mass"]),
    "Dst": float(config_yml["particle"]["Dst"]["mass"]),
    "Psi(4040)": float(params["Psi(4040)_mass"]),
}
psi_width = float(params["Psi(4040)_width"])

# The script overwrites all selected Psi(4040) couplings to 1 + 0j for this isolated probe.
g_total = 1.0 + 0.0j
g_bp_to_psi_k = 1.0 + 0.0j
g_psi_to_dst_d = 1.0 + 0.0j
g_dst_to_d0_pi = 1.0 + 0.0j

print("Hardcoded four-vectors:")
for name, vec in p4.items():
    print(f"  {name:>2}: {vec[0].tolist()}")
print(f"c selector: {c_selector.tolist()}")
print(f"Psi(4040) mass/width: {nominal_mass['Psi(4040)']} / {psi_width}")

Notebook working directory: c:\Users\gamma\Documents\Playground\B2DxDK.jl_playground\Analysis


Loaded config_a.yml, Resonances.yml, and final_params_full.json
Hardcoded four-vectors:
   D: [2.0452, -0.1467, 0.2235, -0.7847]
  D0: [2.2606, 0.2284, -0.3689, 1.2019]
   K: [0.7718, -0.0873, 0.1803, -0.5584]
  pi: [0.2017, 0.0056, -0.0349, 0.1413]
c selector: [-1.0]
Psi(4040) mass/width: 4.039 / 0.08


In [4]:
# -----------------------------------------------------------------------------
# Optional phase-space four-vector generation, copied/adapted from TF-PWA.
# -----------------------------------------------------------------------------
# Source call pattern in the repository:
#   tf-pwa/tf_pwa/applications.py::gen_mc -> PhaseSpaceGenerator(mother, daughters).generate(number)
# Source implementation:
#   tf-pwa/tf_pwa/phasespace.py::PhaseSpaceGenerator
#
# Physical meaning:
#   Generate flat n-body phase-space events for a mother particle decaying at rest
#   into final-state daughters. TF-PWA samples intermediate invariant masses,
#   accepts/rejects them with the phase-space weight prod_i |q_i|, and then
#   recursively performs isotropic two-body decays to obtain daughter four-vectors.
#
# This cell is self-contained NumPy code. It does not import tf_pwa and it does
# not change the hardcoded event used below for the Psi(4040) amplitude check.


def two_body_momentum(m0, m1, m2):
    # Adapted from tf_pwa.phasespace.get_p.
    # Physical meaning: daughter three-momentum magnitude in the parent rest frame.
    m0 = np.asarray(m0, dtype=float)
    m1 = np.asarray(m1, dtype=float)
    m2 = np.asarray(m2, dtype=float)
    p2 = (m0 * m0 - (m1 + m2) ** 2) * (m0 * m0 - (m1 - m2) ** 2)
    p2 = np.where(p2 <= 0.0, 0.0, p2)
    return np.sqrt(p2) / (2.0 * m0)


def neg_spatial_p4(p):
    # TF-PWA LorentzVector.neg keeps energy and flips the spatial momentum.
    p = np.asarray(p, dtype=float)
    return np.concatenate([p[..., 0:1], -p[..., 1:]], axis=-1)


class UniformMassGeneratorNP:
    # NumPy counterpart of tf_pwa.phasespace.UniformGenerator.
    def __init__(self, a, b, rng=None):
        self.a = float(a)
        self.b = float(b)
        self.rng = np.random.default_rng() if rng is None else rng

    def generate(self, n_events):
        return (self.b - self.a) * self.rng.random(n_events) + self.a


class PhaseSpaceGeneratorNP:
    # NumPy counterpart of tf_pwa.phasespace.PhaseSpaceGenerator.
    def __init__(self, mother_mass, daughter_masses, rng=None):
        self.rng = np.random.default_rng() if rng is None else rng
        self.m_mass = []
        self.set_decay(mother_mass, daughter_masses)
        self.sum_mass = sum(self.m_mass)
        self.mass_range = self.get_mass_range()
        self.mass_generator = [None for _ in self.mass_range]

    def get_mass_range(self):
        sm = self.sum_mass - self.m_mass[-1] - self.m_mass[-2]
        m_n = self.m_mass[-1]
        ret = []
        for i in range(self.m_nt - 2):
            b = self.m0 - sm
            a = m_n + self.m_mass[-i - 2]
            ret.append((a, b))
            m_n = a
            sm = sm - self.m_mass[-i - 3]
        return ret

    def set_decay(self, mother_mass, daughter_masses):
        self.m0 = float(mother_mass)
        self.m_nt = len(daughter_masses)
        remaining_kinetic_mass = self.m0
        for mass in daughter_masses:
            self.m_mass.append(float(mass))
            remaining_kinetic_mass -= float(mass)
        if remaining_kinetic_mass <= 0.0:
            raise ValueError("Mother mass is below daughter mass threshold.")

        emmax = remaining_kinetic_mass + self.m_mass[-1]
        emmin = 0.0
        wtmax = 1.0
        for n in range(1, self.m_nt):
            emmin += self.m_mass[-n]
            emmax += self.m_mass[-n - 1]
            wtmax *= two_body_momentum(emmax, emmin, self.m_mass[-n - 1])
        self.m_wtMax = float(wtmax)

    def generate_mass(self, n_events):
        # Sample the nested invariant masses M_i used by the recursive decay.
        sm = self.sum_mass - self.m_mass[-1] - self.m_mass[-2]
        m_n = self.m_mass[-1]
        ret = []
        for i in range(self.m_nt - 2):
            b = self.m0 - sm
            a = m_n + self.m_mass[-i - 2]
            if self.mass_generator[i] is None:
                ms = (b - a) * self.rng.random(n_events) + a
            else:
                ms = self.mass_generator[i].generate(n_events)
            m_n = ms
            sm = sm - self.m_mass[-i - 3]
            ret.append(ms)
        return ret

    def mass_importances(self, masses):
        # Importance correction for non-uniform custom inner-mass generators.
        sm = self.sum_mass - self.m_mass[-1] - self.m_mass[-2]
        m_n = self.m_mass[-1]
        weight = 1.0
        for i, ms in enumerate(masses):
            b = self.m0 - sm
            a = m_n + self.m_mass[-i - 2]
            if i >= 1 and self.mass_generator[i] is None:
                weight = weight * (b - a) / (b - self.mass_range[i][0])
            m_n = ms
            sm = sm - self.m_mass[-i - 3]
        return weight

    def get_weight(self, masses, importances=True):
        # Phase-space acceptance weight proportional to prod_i |q_i| / w_max.
        mass_t = [self.m_mass[-1], *masses, self.m0]
        momenta = []
        for i in range(self.m_nt - 1):
            momenta.append(two_body_momentum(mass_t[i + 1], mass_t[i], self.m_mass[-i - 2]))
        weight = np.prod(np.stack(momenta), axis=0) / self.m_wtMax
        if importances:
            return self.mass_importances(masses) * weight
        return weight

    def flatten_mass(self, masses, importances=True):
        weight = self.get_weight(masses, importances=importances)
        select = weight > self.rng.random(weight.shape)
        return [m[select] for m in masses]

    def generate_momentum_i(self, m0, m1, m2, n_events, p_list=None):
        # Isotropic two-body decay m0 -> m1 + m2, followed by boosting existing
        # descendants into the current rest frame. This mirrors TF-PWA's order.
        if p_list is None:
            p_list = []
        cos_theta = 2.0 * self.rng.random(n_events) - 1.0
        sin_theta = np.sqrt(np.maximum(0.0, 1.0 - cos_theta * cos_theta))
        phi = 2.0 * np.pi * self.rng.random(n_events)
        q = np.broadcast_to(two_body_momentum(m0, m1, m2), phi.shape)

        p = np.stack(
            [
                np.sqrt(q * q + m2 * m2),
                q * sin_theta * np.cos(phi),
                q * sin_theta * np.sin(phi),
                q * cos_theta,
            ],
            axis=-1,
        )
        p_boost = np.stack(
            [
                np.sqrt(q * q + m1 * m1),
                p[:, 1],
                p[:, 2],
                p[:, 3],
            ],
            axis=-1,
        )

        ret = [p]
        if len(p_list) == 0:
            ret.append(neg_spatial_p4(p_boost))
        for old_p in p_list:
            ret.append(rest_vector(p_boost, old_p))
        return ret

    def generate_momentum(self, masses, n_events=None):
        if self.m_nt == 2:
            masses = []
        if n_events is None:
            n_events = len(masses[0]) if masses else 1
        mass_t = [self.m_mass[-1], *masses, self.m0]
        p_list = []
        for i in range(self.m_nt - 1):
            p_list = self.generate_momentum_i(
                mass_t[i + 1], mass_t[i], self.m_mass[-i - 2], n_events, p_list
            )
        return p_list

    def generate(self, n_events, force=True, flatten=True, importances=True):
        # Generate a list of daughter p4 arrays, in the same order as daughter_masses.
        if self.m_nt == 2:
            return self.generate_momentum([], n_events)

        masses = self.generate_mass(n_events)
        if not flatten:
            return self.get_weight(masses, importances=importances), self.generate_momentum(masses, n_events)

        accepted = self.flatten_mass(masses, importances=importances)
        n_generated = len(accepted[0])
        n_total = n_events
        while force and n_generated < n_events:
            guess = int(1.01 * (n_total - n_generated) / (n_generated + 1) * n_events)
            guess = max(1000, min(guess, 4_000_000))
            more = self.flatten_mass(self.generate_mass(guess), importances=importances)
            accepted = [np.concatenate([a, b], axis=0) for a, b in zip(accepted, more)]
            n_generated = len(accepted[0])
            n_total += guess
        if force:
            accepted = [m[:n_events] for m in accepted]
        return self.generate_momentum(accepted)


def generate_b2dxdk_phase_space(n_events, seed=4040):
    # Convenience wrapper for the B+ -> D K D0 pi final state used in config_a.yml.
    # Returned arrays use TF-PWA order [E, px, py, pz].
    rng = np.random.default_rng(seed)
    daughter_order = ["D", "K", "D0", "pi"]
    daughter_masses = [nominal_mass[name] for name in daughter_order]
    generator = PhaseSpaceGeneratorNP(nominal_mass["Bp"], daughter_masses, rng=rng)
    p4_list = generator.generate(n_events)
    return dict(zip(daughter_order, p4_list))


def validate_generated_phase_space(generated_p4, mother_mass, daughter_masses, atol=1e-10):
    # Checks physical consistency of generated phase-space points: on-shell
    # daughters and total four-momentum equal to the mother at rest.
    total = sum(generated_p4.values())
    mass_checks = {
        name: np.max(np.abs(invariant_mass(vec) - daughter_masses[name]))
        for name, vec in generated_p4.items()
    }
    total_target = np.zeros_like(total)
    total_target[:, 0] = mother_mass
    conservation_error = np.max(np.abs(total - total_target))
    passed = conservation_error < atol and all(err < atol for err in mass_checks.values())
    return {
        "passed": bool(passed),
        "max_conservation_error": float(conservation_error),
        "max_mass_errors": {k: float(v) for k, v in mass_checks.items()},
    }


# Phase-Space Generation Flow

In [53]:
print("Phase-space Step 1: Configure the Bp -> D K D0 pi phase-space generator.")
phsp_daughter_order = ["D", "K", "D0", "pi"]
phsp_daughter_masses = {name: nominal_mass[name] for name in phsp_daughter_order}
phsp_generator_demo = PhaseSpaceGeneratorNP(
    nominal_mass["Bp"],
    [phsp_daughter_masses[name] for name in phsp_daughter_order],
    rng=np.random.default_rng(42),
)
print(f"  mother mass: {nominal_mass['Bp']:.8f} GeV")
print("  daughter order:", phsp_daughter_order)
print("  daughter masses:", phsp_daughter_masses)
print("  nested mass ranges:", phsp_generator_demo.mass_range)
print(f"  TF-PWA-style maximum accept/reject weight estimate: {phsp_generator_demo.m_wtMax:.12g}")

Phase-space Step 1: Configure the Bp -> D K D0 pi phase-space generator.
  mother mass: 5.27934000 GeV
  daughter order: ['D', 'K', 'D0', 'pi']
  daughter masses: {'D': 1.86965, 'K': 0.493677, 'D0': 1.86483, 'pi': 0.13957039}
  nested mass ranges: [(2.00440039, 2.916013000000001), (2.4980773899999997, 3.4096900000000008)]
  TF-PWA-style maximum accept/reject weight estimate: 1.30743876115


In [54]:
print("\nPhase-space Step 2: Generate deterministic MC four-vectors with the isolated NumPy copy.")
# The sample is intentionally moderate: enough to compare distributions, but still quick in a notebook.
n_phase_space_validation = 5000
generated_phase_space_example = generate_b2dxdk_phase_space(n_phase_space_validation, seed=42)
print(f"  generated events: {n_phase_space_validation}")
for name in phsp_daughter_order:
    print(f"  first generated {name:>2} four-vector: {generated_phase_space_example[name][0].tolist()}")


Phase-space Step 2: Generate deterministic MC four-vectors with the isolated NumPy copy.
  generated events: 5000
  first generated  D four-vector: [2.1068848914830847, -0.4399095192619785, 0.6915728103380158, 0.5211328873939582]
  first generated  K four-vector: [0.5508659832234902, -0.02701816811087543, 0.17717466951806427, -0.16617913893410244]
  first generated D0 four-vector: [2.160364711984892, 0.19095976652835, -1.070932064865159, -0.07889004944001426]
  first generated pi four-vector: [0.4612244133085344, 0.275967920844504, 0.20218458500907896, -0.2760636990198416]


In [ ]:
print("\nPhase-space Step 3: Validate local generator kinematics.")
generated_phase_space_checks = validate_generated_phase_space(
    generated_phase_space_example,
    nominal_mass["Bp"],
    phsp_daughter_masses,
)
print("  checks:", generated_phase_space_checks)
assert generated_phase_space_checks["passed"], "Generated phase-space points failed mass/conservation checks."
print("  PASS: generated four-vectors are on-shell and conserve four-momentum.")

In [ ]:
print("\nPhase-space Step 4: Compare local generator against real TF-PWA PhaseSpaceGenerator.")
# This imports the linked TF-PWA checkout only for validation; the generator above remains self-contained.
import sys
repo_root = analysis_dir.parent
tfpwa_src = repo_root / "tf-pwa"
if str(tfpwa_src) not in sys.path:
    sys.path.insert(0, str(tfpwa_src))

import tensorflow as tf
from tf_pwa.phasespace import PhaseSpaceGenerator as TFPWAPhaseSpaceGenerator

tf.random.set_seed(4040)
tfpwa_generator = TFPWAPhaseSpaceGenerator(
    nominal_mass["Bp"],
    [phsp_daughter_masses[name] for name in phsp_daughter_order],
)
tfpwa_phase_space_list = [np.asarray(v) for v in tfpwa_generator.generate(n_phase_space_validation)]
tfpwa_phase_space = dict(zip(phsp_daughter_order, tfpwa_phase_space_list))

tfpwa_phase_space_checks = validate_generated_phase_space(
    tfpwa_phase_space,
    nominal_mass["Bp"],
    phsp_daughter_masses,
    atol=1e-7,
)
print("  TF-PWA checks:", tfpwa_phase_space_checks)
assert tfpwa_phase_space_checks["passed"], "TF-PWA phase-space points failed mass/conservation checks."


def pair_mass(sample, names):
    total = sum(sample[name] for name in names)
    return invariant_mass(total)


def distribution_summary(values):
    return np.array([
        np.mean(values),
        np.std(values),
        *np.quantile(values, [0.1, 0.25, 0.5, 0.75, 0.9]),
    ])

comparison_pairs = {
    "m(D,K)": ["D", "K"],
    "m(D0,pi)": ["D0", "pi"],
    "m(D,K,D0)": ["D", "K", "D0"],
}
summary_tolerance = 0.08  # GeV; MC-level tolerance for 5000 independent random events.
for label, names in comparison_pairs.items():
    local_summary = distribution_summary(pair_mass(generated_phase_space_example, names))
    tfpwa_summary = distribution_summary(pair_mass(tfpwa_phase_space, names))
    diff = np.max(np.abs(local_summary - tfpwa_summary))
    print(f"  {label:10s} max summary difference = {diff:.6f} GeV")
    print(f"    local summary: {local_summary}")
    print(f"    TF-PWA summary: {tfpwa_summary}")
    assert diff < summary_tolerance, f"{label} phase-space distribution differs too much from TF-PWA."

print("  PASS: local MC generator agrees with TF-PWA at conservation, on-shell, and distribution-summary levels.")

# Execution Flow

In [27]:
print("Step 1: Reconstruct intermediate four-vectors by summing daughters.")
p4["Dst"] = p4["D0"] + p4["pi"]              # D* -> D0 pi intermediate
p4["Psi(4040)"] = p4["Dst"] + p4["D"]       # Psi(4040) -> D* D intermediate
p4["Bp"] = p4["Psi(4040)"] + p4["K"]        # B+ mother
for name in ["Dst", "Psi(4040)", "Bp"]:
    print(f"  {name:>9} p4 = {p4[name][0].tolist()}")

Step 1: Reconstruct intermediate four-vectors by summing daughters.
        Dst p4 = [2.4623, 0.23399999999999999, -0.4038, 1.3432]
  Psi(4040) p4 = [4.5075, 0.08729999999999999, -0.1803, 0.5585]
         Bp p4 = [5.2793, -1.3877787807814457e-17, 0.0, 9.999999999998899e-05]


In [28]:
print("\nStep 2: Compute invariant masses from the event kinematics.")
event_mass = {name: invariant_mass(vec) for name, vec in p4.items()}
for name in ["Bp", "Psi(4040)", "Dst", "D", "K", "D0", "pi"]:
    print(f"  m({name}) = {event_mass[name][0]:.12f} GeV")


Step 2: Compute invariant masses from the event kinematics.
  m(Bp) = 5.279299999053 GeV
  m(Psi(4040)) = 4.468277589855 GeV
  m(Dst) = 2.010205116400 GeV
  m(D) = 1.869656602160 GeV
  m(K) = 0.493695553960 GeV
  m(D0) = 1.864804273912 GeV
  m(pi) = 0.139527165814 GeV


In [29]:
print("\nStep 3: Compute helicity Euler angles for the sequential decay chain.")
chain = [
    ("Bp", ["Psi(4040)", "K"]),
    ("Psi(4040)", ["Dst", "D"]),
    ("Dst", ["D0", "pi"]),
]
angles = calculate_helicity_angles(p4, chain)
for core, outs in chain:
    for out in outs:
        a = angles[core][out]
        print(
            f"  {core:>9} -> {out:<9}: "
            f"alpha={a['alpha'][0]: .12f}, beta={a['beta'][0]: .12f}, gamma={a['gamma'][0]: .12f}"
        )


Step 3: Compute helicity Euler angles for the sequential decay chain.
         Bp -> Psi(4040): alpha=-1.119874085778, beta= 0.344435772294, gamma= 0.000000000000
         Bp -> K        : alpha=-4.261466739368, beta= 2.797156881296, gamma= 0.000000000000
  Psi(4040) -> Dst      : alpha= 1.990557934214, beta= 0.034250379703, gamma= 0.000000000000
  Psi(4040) -> D        : alpha=-1.151034719376, beta= 3.107342273887, gamma= 0.000000000000
        Dst -> D0       : alpha=-1.215919835627, beta= 2.613576784812, gamma= 0.000000000000
        Dst -> pi       : alpha=-4.357512489216, beta= 0.528015868778, gamma= 0.000000000000


In [30]:
print("\nStep 4: Compute breakup momenta q^2 and nominal q0^2 for barrier factors.")
q2_bp = get_relative_p2(event_mass["Bp"], event_mass["Psi(4040)"], event_mass["K"])
q02_bp = get_relative_p2(nominal_mass["Bp"], nominal_mass["Psi(4040)"], nominal_mass["K"])
q2_psi = get_relative_p2(event_mass["Psi(4040)"], event_mass["Dst"], event_mass["D"])
q02_psi = get_relative_p2(nominal_mass["Psi(4040)"], nominal_mass["Dst"], nominal_mass["D"])
q2_dst = get_relative_p2(event_mass["Dst"], event_mass["D0"], event_mass["pi"])
print(f"  Bp -> Psi K:      q2={q2_bp[0]:.12f}, q02={q02_bp:.12f}")
print(f"  Psi -> Dst D:     q2={q2_psi[0]:.12f}, q02={q02_psi:.12f}")
print(f"  Dst -> D0 pi:     q2={q2_dst[0]:.12f}; no barrier factor in this config")


Step 4: Compute breakup momenta q^2 and nominal q0^2 for barrier factors.
  Bp -> Psi K:      q2=0.351956267230, q02=1.005576573382
  Psi -> Dst D:     q2=1.226829388171, q02=0.314573138441
  Dst -> D0 pi:     q2=0.001552696344; no barrier factor in this config


In [31]:
print("\nStep 5: Build the three helicity-decay tensors.")
# Spin bases match TF-PWA: spin-0 -> (0,), spin-1 -> (-1, 0, 1).
spin0 = (0,)
spin1 = (-1, 0, 1)

amp_bp = helicity_decay_amp(
    core_j=0,
    out_js=(1, 0),
    core_spins=spin0,
    out_spins=(spin1, spin0),
    ls_list=((1, 1),),
    g_ls=g_bp_to_psi_k,
    angle=angles["Bp"]["Psi(4040)"],
    mass_core=event_mass["Bp"],
    q2=q2_bp,
    q02=q02_bp,
    has_barrier=True,
    barrier_norm=True,
)
print(f"  Bp -> Psi K tensor shape {amp_bp.shape}; flat = {amp_bp.reshape(-1)}")

amp_psi = helicity_decay_amp(
    core_j=1,
    out_js=(1, 0),
    core_spins=spin1,
    out_spins=(spin1, spin0),
    ls_list=((1, 1),),
    g_ls=g_psi_to_dst_d,
    angle=angles["Psi(4040)"]["Dst"],
    mass_core=event_mass["Psi(4040)"],
    q2=q2_psi,
    q02=q02_psi,
    has_barrier=True,
    barrier_norm=True,
)
print(f"  Psi -> Dst D tensor shape {amp_psi.shape}; flat = {amp_psi.reshape(-1)}")

amp_dst = helicity_decay_amp(
    core_j=1,
    out_js=(0, 0),
    core_spins=spin1,
    out_spins=(spin0, spin0),
    ls_list=((1, 0),),
    g_ls=g_dst_to_d0_pi,
    angle=angles["Dst"]["D0"],
    mass_core=event_mass["Dst"],
    q2=q2_dst,
    q02=np.array([0.0]),
    has_barrier=False,
    barrier_norm=False,
)
print(f"  Dst -> D0 pi tensor shape {amp_dst.shape}; flat = {amp_dst.reshape(-1)}")


Step 5: Build the three helicity-decay tensors.
  Bp -> Psi K tensor shape (1, 1, 3, 1); flat = [ 0.        +0.j -0.91871445+0.j  0.        +0.j]
  Psi -> Dst D tensor shape (1, 3, 3, 1); flat = [-3.20913124e-01-7.19074007e-01j  0.00000000e+00-0.00000000e+00j
  9.41332786e-05+2.10925602e-04j -1.90724787e-02+0.00000000e+00j
  0.00000000e+00+0.00000000e+00j -1.90724787e-02+0.00000000e+00j
 -9.41332786e-05+2.10925602e-04j  0.00000000e+00+0.00000000e+00j
  3.20913124e-01-7.19074007e-01j]
  Dst -> D0 pi tensor shape (1, 3, 1, 1); flat = [ 0.12378949+0.33405639j -0.86380842+0.j         -0.12378949+0.33405639j]


In [32]:
print("\nStep 6: Compute particle factors.")
q_psi = np.sqrt(q2_psi)
q0_psi = np.sqrt(q02_psi)
psi_factor = bwr(event_mass["Psi(4040)"], nominal_mass["Psi(4040)"], psi_width, q_psi, q0_psi, l=1, d=3.0)
dst_factor = np.ones_like(psi_factor, dtype=complex)  # Dst uses model: one
print(f"  Psi(4040) C(BWR) factor = {format_complex(psi_factor)}")
print(f"  Dst model-one factor     = {format_complex(dst_factor)}")
print("  Since Psi(4040) has C=-1 in config_a.yml, the local C(BWR) wrapper passes the BWR through unchanged.")


Step 6: Compute particle factors.
  Psi(4040) C(BWR) factor = -0.2636956535053299 + 0.05167925958906836j
  Dst model-one factor     = 1 + 0j
  Since Psi(4040) has C=-1 in config_a.yml, the local C(BWR) wrapper passes the BWR through unchanged.


In [33]:
print("\nStep 7: Contract the tensors exactly like DecayChain.get_amp.")
# TF-PWA uses the equivalent einsum string:
#   ...agd,...gfb,...fce,...->...abcde
# Here:
#   a = Bp spin index, g = Psi spin index, d = K spin index,
#   f = Dst spin index, b = D spin index, c = D0 spin index, e = pi spin index.
particle_factor = g_total * psi_factor * dst_factor
amplitude_tensor = np.einsum("...agd,...gfb,...fce,...->...abcde", amp_bp, amp_psi, amp_dst, particle_factor)
amplitude = amplitude_tensor.reshape(-1)[0]
print(f"  amplitude tensor shape = {amplitude_tensor.shape}")
print(f"  independent Psi(4040) amplitude = {format_complex(amplitude)}")


Step 7: Contract the tensors exactly like DecayChain.get_amp.
  amplitude tensor shape = (1, 1, 1, 1, 1, 1)
  independent Psi(4040) amplitude = -0.0006049977356379838 - 0.003087027069212293j


In [34]:
print("\nStep 8: Numerical regression check against the TF-PWA reference value for this isolated component.")
# This constant is the value obtained from TF-PWA's dg.get_amp(phsp_variables) for chain index 5 with the same
# p_unit parameter manipulation. It is not used in the calculation above; it is only a regression check.
tfpwa_reference = complex(-0.0006049977354135942, -0.0030870270680673287)
delta = amplitude - tfpwa_reference
print(f"  TF-PWA reference amplitude = {tfpwa_reference.real:.16g} - {abs(tfpwa_reference.imag):.16g}j")
print(f"  difference                 = {delta.real:.3e} + {delta.imag:.3e}j")
assert abs(delta) < 2e-12, "Independent implementation does not match TF-PWA reference closely enough."
print("  PASS: independent notebook implementation matches TF-PWA within tolerance.")


Step 8: Numerical regression check against the TF-PWA reference value for this isolated component.
  TF-PWA reference amplitude = -0.0006049977354135942 - 0.003087027068067329j
  difference                 = -2.244e-13 + -1.145e-12j
  PASS: independent notebook implementation matches TF-PWA within tolerance.


# Comparison to TF-PWA

In [23]:
# Optional: rerun this cell if you want a live comparison against the installed TF-PWA checkout.
# The independent calculation above does not use tf_pwa; this cell is only a verification aid.
import os
import sys

repo_root = analysis_dir.parent
sys.path.insert(0, str(repo_root / "tf-pwa"))
sys.path.insert(0, str(analysis_dir))
os.chdir(analysis_dir)

import tensorflow as tf
import extra_amp  # registers local C(BWR), C(one), etc.
from tf_pwa.config_loader import ConfigLoader

config = ConfigLoader("config_a.yml")
with open("final_params_full.json", "r", encoding="utf-8") as f:
    params_dict = json.load(f)["value"]

particles = list(config.get_decay().outs)
particle_map = {p.name: p for p in particles}
p4_tfpwa = {
    particle_map["D"]: tf.constant([[2.0452, -0.1467, 0.2235, -0.7847]], dtype=tf.float64),
    particle_map["D0"]: tf.constant([[2.2606, 0.2284, -0.3689, 1.2019]], dtype=tf.float64),
    particle_map["K"]: tf.constant([[0.7718, -0.0873, 0.1803, -0.5584]], dtype=tf.float64),
    particle_map["pi"]: tf.constant([[0.2017, 0.0056, -0.0349, 0.1413]], dtype=tf.float64),
}
phsp_variables = config.data.cal_angle(p4_tfpwa)
phsp_variables["c"] = np.array([-1.0])

amp_model = config.get_amplitude()
dg = amp_model.decay_group
chain = dg.chains[5]

p_unit = params_dict.copy()
for key in p_unit:
    if ("total" in key or "g_ls" in key) and (key.endswith("r") or key.endswith("i")):
        p_unit[key] = 0.0
for d_idx, decay in enumerate(chain.chain):
    prefix = f"{decay.core.name.replace('(1+)', '(1.)')}->{'.'.join([p.name.replace('(1+)', '(1.)') for p in decay.outs])}"
    for key in p_unit:
        if prefix in key and ("total" in key or "g_ls" in key) and key.endswith("_0r"):
            p_unit[key] = 1.0

config.set_params(p_unit)
dg.set_used_chains([5])
tfpwa_live = dg.get_amp(phsp_variables).numpy().reshape(-1)[0]
print(f"Independent amplitude: {format_complex(amplitude)}")
print(f"Live TF-PWA amplitude: {format_complex(tfpwa_live)}")
print(f"Live difference: {amplitude - tfpwa_live}")

ModuleNotFoundError: No module named 'tensorflow'